In [1]:
import pandas as pd
import requests
import os
import re
from bs4 import BeautifulSoup

In [2]:
metadata_table_long = pd.read_csv("../data/metadata_table_long.csv")

In [6]:
metadata_table_long.head(5)

,Author,Full title,In,Year,Place,Publisher/Printer,Era,Form/Genre,Discipline/Content,Original,...,Written by,Library and Signature,ids,id,date_min,date_max,filename,file_year,authors_gnds,author_wd
0,"Achrelius, Daniel",Scientiarum magnes recitatus publice anno 1690...,NaN,1690,[Turku],Wall,17th century,Oration,"Mathematics, Astronomy/Astrology/Cosmography, ...",Scientiarum magnes(Google Books),...,IT,NaN,[705665],705665,1690.0,1690.0,"Achrelius,_Daniel_-_Scientiarum_magnes__Turku_...",1690.0,['1055219730'],Q5541951
1,"Acidalius, Valens","Ad Iordanum Brunum Nolanum, Italum","Poematum Iani Lernutii, Iani Gulielmi, Valenti...",1603,"Liegnitz, Wrocław","Albert, David",17th century,Panegyric poem,Astronomy/Astrology/Cosmography,Ad Iordanum Brunum (1603)(CAMENA)Ad Iordanum B...,...,MK,NaN,[801745],801745,1603.0,1603.0,Janus_Lernutius_et_al__-_Poemata__Liegnitz_160...,1603.0,[],NaN
2,"Acosta, José de",De natura novi orbis libri duo et De promulgat...,NaN,1589,Salamanca,Guillelum Foquel,16th century,Monograph,"Astronomy/Astrology/Cosmography, Geography/Car...",De natura novi orbis(Biodiversity Heritage Lib...,...,DB,NaN,[713323],713323,1589.0,1589.0,"Acosta,_José_de_-_De_natura_novi_orbis__Salama...",1589.0,['12116117X'],Q555476
3,"Adam, Melchior","Vitae Germanorum medicorum, qui saeculo superi...",NaN,1620,Heidelberg,"Rosa, Geyder",17th century,Biography,Medicine,Vitae Germanorum medicorum(MDZ)Alternative lin...,...,IT,NaN,[693148],693148,1620.0,1620.0,"Adam,_Melchior_-_Vitae_Germanorum_medicorum__H...",1620.0,['12005860X'],Q67082
4,"Addison, Joseph",Ad insignissimum virum dominum Thomam Burnettu...,"Examen poeticum duplex, sive, Musarum anglican...",1698,London,Richard Wellington I.,17th century,Panegyric poem,Meteorology/Earth sciences,Ad Burnettum sacrae theoriae telluris auctorem...,...,MK,NaN,[769230],769230,1698.0,1698.0,Examen_poeticum_duplex__London_1698_pdf.txt,1698.0,['118500619'],Q206384


In [3]:
resp = requests.get("https://wiki.uibk.ac.at/noscemus/Nicander_Colophonius_Lonicer")
soup = BeautifulSoup(resp.content, "html.parser")
table = soup.find("table", class_="wikitable")
table

In [4]:
base_url = "https://wiki.uibk.ac.at/noscemus/" # "https://wiki.uibk.ac.at/noscemus/"

In [5]:
# get authors from filename - does not work pefectly, since there are collected volumes with filename starting by someone else than author...
def get_authors_list(filename):
    # remove file extensions
    fn = filename.replace(".pdf.txt", "").replace(".txt", "")

    # if there is no "_-_" delimiter → no authors
    if "_-_" not in fn:
        return []

    # split authors part from the rest
    authors_str, _, _ = fn.partition("_-_")

    # split multiple authors
    authors = authors_str.split("_&_")

    # strip stray underscores at edges but KEEP internal ones
    authors = [a.strip("_") for a in authors]
    authors = [a.replace("_et_al", "") for a in authors]
    # do not discard names containing no comma!
    # (Because e.g. "Varii" is a valid collective author)
    return authors

In [76]:
authors_list_series = metadata_table_long["filename"].apply(get_authors_list)

In [77]:
authors_list_series.sample(50).tolist()

[['Stupanus,_Johann_Niklaus'],
 ['Leopold,_Christoph', 'Vian,_Christoph'],
 ['Boerhaave,_Herman'],
 ['Schurzfleisch,_Konrad_Sameul', 'Hofmann,_Johann_Georg'],
 ['Cappeller,_Moritz_Anton'],
 ['Hippocrates', 'Leonberger,_Georg'],
 ['Savastano,_Francesco_Eulalio'],
 ['Drechsler,_Johann_Gabriel', 'Rechtenbach,_Polycarb_Michael'],
 ['Scaliger,_Joseph_Juste'],
 ['Bauhin,_Caspar'],
 ['Aselli,_Gaspare'],
 ['Merian,_Maria_Sibylla'],
 [],
 ['Newton,_Isaac'],
 ['Caius,_John'],
 ['Marggraff,_Michael', 'Bleidner,_Johannes'],
 ['Nardi,_Giovanni'],
 ['Du_Hamel,_Jean-Baptiste'],
 ['Willughby,_Francis', 'Ray,_John'],
 [],
 ['Ps_-Proclus', 'Linacre,_Thomas'],
 ['Scheiner,_Christoph'],
 ['Riccioli,_Giovanni_Battista'],
 ['Schott,_Kaspar'],
 ['Müller,_Otto'],
 ['Melancthon,_Philipp', 'Milch,_Jacob'],
 ['Reuss,_Jeremias_David'],
 ['Dürer,_Albrecht', 'Camerarius,_Joachim'],
 ['Pereira,_Benedict'],
 ['Taruffi,_Giuseppe_Antonio'],
 ['Boncompagni,_Baldassarre', 'Leonardus,_Pisanus'],
 ['Unger,_Bernhard'],
 ['H

In [114]:
import re

def parse_authors(s):
    if not isinstance(s, str) or not s.strip():
        return []

    # Rule 1: cut at " ("
    s = s.split(" (")[0]

    # Rule 2: remove "et al"
    s = re.sub(r"\bet\s+al\b\.?", "", s, flags=re.IGNORECASE)

    # Rule 3: split only at comma + capital letter
    parts = re.split(r",(?=[A-Z])", s)

    # trim each part
    parts = [p.strip() for p in parts if p.strip()]

    # Rule 4: replace spaces with underscores
    parts = [p.replace(" ", "_") for p in parts]

    return parts

In [118]:
authors_list_series = metadata_table_long["Author"].apply(parse_authors)
authors_list_series.sample(10, random_state=100)

641                               [Morhof,_Daniel_Georg]
747    [Quelmalz,_Samuel_Theodor, Crell,_Johann_Fried...
222                                    [Clüver,_Philipp]
825                             [Scaliger,_Joseph_Juste]
943                         [Vorstius,_Aelius_Everardus]
320                                  [Froidmont,_Libert]
650                   [Müller_von_Löwenstein,_Friedrich]
413                                [Hartsoeker,_Nicolas]
553                 [Linden,_Johannes_Antonides_van_der]
178                                       [Béguin,_Jean]
Name: Author, dtype: object

In [120]:
def extract_gnd(authors_list, base_url=base_url):
    gnds = []
    for a in authors_list:
        url = os.path.join(base_url, a)
        try:
            resp = requests.get(url)
            resp.raise_for_status()

            soup = BeautifulSoup(resp.content, "html.parser")
            table = soup.find("table", class_="wikitable")
            # Regex for any GND ID: digits optionally followed by a capital letter
            gnd_regex = re.compile(r"gnd/([0-9]+[0-9X])")  # also matches IDs ending in X

            for row in table.find_all("tr"):
                header = row.find("th")
                data = row.find("td")
                if not header or not data:
                    continue

                # Detect any row referring to GND in its header
                if "gnd" in header.text.lower():
                    # Look for the GND-containing URL
                    link = data.find("a", href=re.compile("gnd/"))
                    if link:
                        gnd_url = link["href"]
                        m = gnd_regex.search(gnd_url)
                        if m:
                            gnds.append(m.group(1))  # return just the ID (alphanumeric)
        except:
            pass
    return gnds

In [ ]:
authors_gnds = authors_list_series.apply(extract_gnd)

In [128]:
authors_gnds.sample(10)

106               [131501259]
626               [118581201]
747    [100540686, 121631524]
272    [11852786X, 118518569]
692               [102905444]
615               [115667156]
816               [13387060X]
860                        []
581               [118930095]
127               [117621455]
Name: Author, dtype: object

In [ ]:
metadata_table_long["authors_gnds"] = authors_gnds

In [129]:

def wd_by_an_identifier(property, idstring):
    """
    Query Wikidata for an entity with the given TLG author ID (property P1266).
    Returns JSON results from the WDQS endpoint.
    """
    query = f"""
    SELECT ?item ?itemLabel WHERE {{
      ?item wdt:{property} "{idstring}" .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    url = "https://query.wikidata.org/sparql"
    headers = {"Accept": "application/sparql-results+json"}

    response = requests.get(url, params={"query": query, "format": "json"}, headers=headers)
    response.raise_for_status()

    return response.json()



In [134]:
wd_by_an_identifier("P227", "131501259")["results"]["bindings"][0]["item"]["value"].rpartition("/")[2]

'Q1419869'

In [137]:
def get_wd_authors(gnds_list):
    wds = []
    for gnd in gnds_list:
        try:
            wd_id =  wd_by_an_identifier("P227", gnd)["results"]["bindings"][0]["item"]["value"].rpartition("/")[2]
            wds.append(wd_id)
        except:
            pass
    return ", ".join(wds)

In [140]:
author_wikidata_ids = metadata_table_long["authors_gnds"].apply(get_wd_authors)
author_wikidata_ids.head(10)

0    Q5541951
1            
2     Q555476
3      Q67082
4     Q206384
5     Q206384
6     Q345654
7     Q571155
8      Q76579
9      Q76579
Name: authors_gnds, dtype: object

In [141]:
metadata_table_long["author_wd"] = author_wikidata_ids

In [8]:
headers = {
        "User-Agent": "MyResearchBot/0.1 (kase@ff.zcu.cz)"
    }

def get_claim_value(qid, pid, headers=headers):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": qid,
        "props": "claims",
        "format": "json"
    }
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()
    data = r.json()

    claims = data["entities"][qid]["claims"]

    if pid not in claims:
        return None  # property not present

    # take the first value
    mainsnak = claims[pid][0]["mainsnak"]
    datavalue = mainsnak.get("datavalue", {})
    return datavalue.get("value")

get_claim_value("Q9047", "P7935")

'Leibniz_Gottfried_Wilhelm'

In [9]:
get_claim_value("Q9047", "P214")

'9849392'

In [17]:
def get_author_viaf(wds_string_list):
    wds_list = str(wds_string_list).split(", ")
    viafs = []
    for wd in wds_list:
        try:
            viaf = get_claim_value(wd, "P214")
            viafs.append(viaf)
        except:
            pass
    return ", ".join(viafs)

In [15]:
for wd_string_list in metadata_table_long["author_wd"][:10]:
    print(get_author_viaf(wd_string_list))

['44659002']
[]
['2465365']
['29559128']
['7413288']
['7413288']
['17292983']
['3217482']
['100164815']
['100164815']


In [18]:
metadata_table_long["author_viaf"] = metadata_table_long["author_wd"].apply(get_author_viaf)

In [38]:
metadata_table_long["author_gnd"] = metadata_table_long["authors_gnds"].apply(lambda x: ", ".join(eval(x)))

In [41]:
metadata_table_long.drop("authors_gnds", axis=1, inplace=True)

In [42]:
metadata_table_long.head(5)

,Author,Full title,In,Year,Place,Publisher/Printer,Era,Form/Genre,Discipline/Content,Original,...,Library and Signature,ids,id,date_min,date_max,filename,file_year,author_wd,author_viaf,author_gnd
0,"Achrelius, Daniel",Scientiarum magnes recitatus publice anno 1690...,NaN,1690,[Turku],Wall,17th century,Oration,"Mathematics, Astronomy/Astrology/Cosmography, ...",Scientiarum magnes(Google Books),...,NaN,[705665],705665,1690.0,1690.0,"Achrelius,_Daniel_-_Scientiarum_magnes__Turku_...",1690.0,Q5541951,44659002,1055219730
1,"Acidalius, Valens","Ad Iordanum Brunum Nolanum, Italum","Poematum Iani Lernutii, Iani Gulielmi, Valenti...",1603,"Liegnitz, Wrocław","Albert, David",17th century,Panegyric poem,Astronomy/Astrology/Cosmography,Ad Iordanum Brunum (1603)(CAMENA)Ad Iordanum B...,...,NaN,[801745],801745,1603.0,1603.0,Janus_Lernutius_et_al__-_Poemata__Liegnitz_160...,1603.0,NaN,,
2,"Acosta, José de",De natura novi orbis libri duo et De promulgat...,NaN,1589,Salamanca,Guillelum Foquel,16th century,Monograph,"Astronomy/Astrology/Cosmography, Geography/Car...",De natura novi orbis(Biodiversity Heritage Lib...,...,NaN,[713323],713323,1589.0,1589.0,"Acosta,_José_de_-_De_natura_novi_orbis__Salama...",1589.0,Q555476,2465365,12116117X
3,"Adam, Melchior","Vitae Germanorum medicorum, qui saeculo superi...",NaN,1620,Heidelberg,"Rosa, Geyder",17th century,Biography,Medicine,Vitae Germanorum medicorum(MDZ)Alternative lin...,...,NaN,[693148],693148,1620.0,1620.0,"Adam,_Melchior_-_Vitae_Germanorum_medicorum__H...",1620.0,Q67082,29559128,12005860X
4,"Addison, Joseph",Ad insignissimum virum dominum Thomam Burnettu...,"Examen poeticum duplex, sive, Musarum anglican...",1698,London,Richard Wellington I.,17th century,Panegyric poem,Meteorology/Earth sciences,Ad Burnettum sacrae theoriae telluris auctorem...,...,NaN,[769230],769230,1698.0,1698.0,Examen_poeticum_duplex__London_1698_pdf.txt,1698.0,Q206384,7413288,118500619


In [43]:
metadata_table_long.to_csv("../data/metadata_table_long.csv", index=False)

In [44]:
!cp ../data/metadata_table_long.csv /srv/data/tome/noscemus/metadata_table_long.csv
